In [1]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content", "filename"],
    keyword_fields=[]
)

index.fit(parsed_docs)

In [2]:
from minsearch import Highlighter, Tokenizer

In [5]:
query = "How do I use evidently to monitor my machine learning models?"

search_results = index.search(
    query=query,
    num_results=5
)

highlighter = Highlighter(
    highlight_fields=["content"]
)

highlighted_results = highlighter.highlight(
    query,
    search_results
)

highlighted_results

[{'title': 'Use HuggingFace models',
  'description': 'How to use models from HuggingFace as evaluators.',
  'content': {'matches': ['You can score your text by downloading and using ML **models** from HuggingFace. This lets you apply any criteria from the source model, e.g. classify texts by em...',
    '...s you apply any criteria from the source model, e.g. classify texts by emotion. There are:\n\n* Ready-**to**-**use** descriptors that wrap a specific model,\n\n* A general interface **to** call other suitable **models** yo...',
    '...ou apply any criteria from the source model, e.g. classify texts by emotion. There are:\n\n* Ready-**to**-**use** descriptors that wrap a specific model,\n\n* A general interface **to** call other suitable **models** you se...',
    '... emotion. There are:\n\n* Ready-**to**-**use** descriptors that wrap a specific model,\n\n* A general interface **to** call other suitable **models** you select.\n\n**Pre-requisites**:\n\n* You know **how** **to** **u

In [6]:
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(
        stemmer="snowball",
        stop_words=stopwords
    )
)

In [7]:
highlighted_results = highlighter.highlight(
    query,
    search_results
)

highlighted_results

[{'title': 'Use HuggingFace models',
  'description': 'How to use models from HuggingFace as evaluators.',
  'content': {'matches': ['...downloading and using ML **models** from HuggingFace. This l...',
    '...criteria from the source **model**, e.g. classify texts by ...',
    '.... There are:\n\n* Ready-to-**use** descriptors that wrap a ...'],
   'total_matches': 33},
  'filename': 'metrics/customize_hf_descriptor.mdx'},
 {'title': 'Evidently Cloud',
  'description': 'How to set up Evidently Cloud account.',
  'icon': 'cloud',
  'content': {'matches': ['... and Evidently Cloud v1, **use** `from evidently.ui.works...'],
   'total_matches': 1},
  'filename': 'docs/setup/cloud.mdx'},
 {'title': 'Why Evidently?',
  'description': 'Why choose Evidently.',
  'content': {'matches': ['...ucts: whether it’s an ML **model**, an LLM app, or a comple...',
    '...workflow.\n\nOur tools are **model**-, framework-, and applic...',
    '.... It’s up to you.\xa0\n- Add **monitoring** as you are re

In [8]:
file_index = {
    doc["filename"]: doc["content"]
    for doc in parsed_docs
}

In [9]:
from typing import Any


class SearchTools:

    def __init__(self, index: Any, highlighter: Any, file_index: dict):
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str):
        """
        Search the documentation and return relevant snippets.

        Args:
            query: The search query.
        """
        search_results = self.index.search(
            query=query,
            num_results=5
        )

        return self.highlighter.highlight(
            query,
            search_results
        )

    def get_file(self, filename: str) -> str:
        """
        Return the full content of a documentation file.

        Args:
            filename: The filename to retrieve.
        """
        if filename in self.file_index:
            return self.file_index[filename]

        return f"File {filename} does not exist."

In [10]:
search_tools = SearchTools(
    index=index,
    highlighter=highlighter,
    file_index=file_index
)

results = search_tools.search(query)
results

[{'title': 'Use HuggingFace models',
  'description': 'How to use models from HuggingFace as evaluators.',
  'content': {'matches': ['...downloading and using ML **models** from HuggingFace. This l...',
    '...criteria from the source **model**, e.g. classify texts by ...',
    '.... There are:\n\n* Ready-to-**use** descriptors that wrap a ...'],
   'total_matches': 33},
  'filename': 'metrics/customize_hf_descriptor.mdx'},
 {'title': 'Evidently Cloud',
  'description': 'How to set up Evidently Cloud account.',
  'icon': 'cloud',
  'content': {'matches': ['... and Evidently Cloud v1, **use** `from evidently.ui.works...'],
   'total_matches': 1},
  'filename': 'docs/setup/cloud.mdx'},
 {'title': 'Why Evidently?',
  'description': 'Why choose Evidently.',
  'content': {'matches': ['...ucts: whether it’s an ML **model**, an LLM app, or a comple...',
    '...workflow.\n\nOur tools are **model**-, framework-, and applic...',
    '.... It’s up to you.\xa0\n- Add **monitoring** as you are re

In [11]:
search_tools.get_file("docs/setup/cloud.mdx")

'<Note>\n  Evidently Cloud is no longer available as a SaaS product, but you can self-host the open-source Evidently Platform or continue using the Evidently library integrated with other tools.\n</Note>\n\n## 1. Create an Account\n\n- If not yet, sign up for a [free Evidently Cloud account](https://app.evidently.cloud/signup).\n- After logging in, create an **Organization** and name it.\n\n## 2. Connect from Python\n\n<Info>\n  You need this for programmatic tasks like tracing or logging local evals. Many other tasks can be done directly on the platform.\n</Info>\n\n### Get a Token\n\nClick the **Key** menu icon to open the [Token page](https://app.evidently.cloud/token). Generate and save token securely.\n\n### Install Evidently\n\n[Install](/docs/setup/installation) the Evidently Python library.\n\n```python\npip install evidently ## or pip install evidently[llm]\n```\n\n### Connect\n\nImport the cloud workspace and pass your API token to connect:\n\n```python\nfrom evidently.ui.wor

In [12]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tools(search_tools)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'get_file',
  'description': 'Return the full content of a documentation file.\n\nArgs:\n    filename: The filename to retrieve.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'}},
   'required': ['filename'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the documentation and return relevant snippets.\n\nArgs:\n    query: The search query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [13]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base.

First, search the documentation for relevant information.

Analyze the search results and identify the most relevant files.

Use get_file to retrieve the full content of the relevant files.

Read and analyze the full files before producing the final answer.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to
include the word 'evidently' in search queries.
"""

In [14]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=llm_client
)

In [15]:
result = agent.loop(
    "How do I create an Evidently dashboard?"
)

print(result.last_message)

To create a dashboard in Evidently, follow these steps based on the method you prefer: using the user interface (UI) or the Python API.

### Creating a Dashboard through the User Interface (UI)
1. **Access the Dashboard**:
   - Navigate to your project in Evidently Cloud or Enterprise.

2. **Add a New Tab**:
   - Enter "Edit" mode on your Dashboard.
   - Click the plus sign to "Add Tab."
   - Choose to create an "empty" tab or start with a pre-built template.

3. **Add Panels**:
   - Click on the "Add Panel" button.
   - Configure your panel by selecting metrics, filtering by tags or metrics labels, and specifying the visual type (e.g., text, counter, pie chart, line plot).
   - Preview your setup and click "Save" to add the panel to your chosen tab.

4. **Panel Types**:
   - You can create various panel types including counters, pie charts, line plots, and bar plots.
   - Ensure you have relevant reports with data metrics available for the panels to show meaningful information.

### C

In [16]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

chat_interface = IPythonChatInterface()

runner_callback = DisplayingRunnerCallback(
    chat_interface=chat_interface
)

In [17]:
result = agent.loop(
    "How do I create an Evidently dashboard?",
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received
